In [1]:
import numpy as np
import pandas as pd

In [3]:
cases = pd.read_csv("./data/data1k.csv")

In [5]:
cases.head().T

,0,1,2,3,4
person_id,199031000000439,15100000000316,15100000000316,10210000000095,51220000000305
HearingDate,2018-06-01,2000-08-07,2000-08-07,2019-09-25,2010-05-03
CodeSection,A.46.2-862,18.2-26,18.2-95,46.2-300,46.2-613(2)
codesection,covered elsewhere,covered elsewhere,covered elsewhere,covered elsewhere,covered elsewhere
ChargeType,Misdemeanor,Felony,Felony,Misdemeanor,Misdemeanor
chargetype,Misdemeanor,Felony,Felony,Misdemeanor,Misdemeanor
Class,NaN,NaN,NaN,NaN,1
DispositionCode,Guilty,Guilty,Guilty,Guilty In Absentia,Guilty
disposition,Conviction,Conviction,Conviction,Conviction,Conviction
Plea,NaN,NaN,NaN,NaN,Guilty


What code sections are most frequent?

In [6]:
cases['CodeSection'].value_counts().head(15)

CodeSection
B.46.2-301    268
A.46.2-862    253
46.2-300      177
C.46.2-862    123
18.2-250.1     96
18.2-95        71
A.18.2-266     68
18.2-57        68
A.46.2-852     62
18.2-250       58
18.2-96        53
A.46.2-853     51
A.46.2-707     48
18.2-388       45
18.2-172       44
Name: count, dtype: int64

Which ones most often lead to convictions?

In [7]:
cases['DispositionCode'].value_counts()

DispositionCode
Guilty                  1514
Nolle Prosequi           484
Dismissed                443
Guilty In Absentia       295
Not Guilty                57
Not Guilty/Acquitted      17
Dismissed/Other            1
Name: count, dtype: int64

In [10]:
cases['Conviction'] = [x in ['Guilty', 'Guilty In Absentia'] for x in cases['DispositionCode']]
cases['Conviction'].value_counts()

Conviction
True     1809
False    1002
Name: count, dtype: int64

In [16]:
cases_convict_rate = cases.groupby('CodeSection').agg({'Conviction': ['mean', 'count']}).reset_index()
cases_convict_rate.columns = ['CodeSection', 'convict_rate', 'count']
cases_convict_rate = cases_convict_rate.query("count >= 30").sort_values('convict_rate', ascending=False)
cases_convict_rate


,CodeSection,convict_rate,count
317,A.46.2-862,0.924901,253
334,C.46.2-862,0.886179,123
303,A.18.2-266,0.852941,68
242,46.2-300,0.751412,177
325,B.46.2-301,0.746269,268
114,18.2-388,0.733333,45
63,18.2-248,0.707317,41
312,A.46.2-852,0.693548,62
154,18.2-91,0.684211,38
313,A.46.2-853,0.627451,51


Which ones have the most severe racial disparities?

In [17]:
cases['Race'].value_counts()

Race
White Caucasian(Non-Hispanic)                  1093
Black(Non-Hispanic)                             864
White Caucasian (Non-Hispanic)                  342
Black (Non-Hispanic)                            311
Hispanic                                         88
Asian Or Pacific Islander                        27
Other(Includes Not Applicable.. Unknown)         27
White                                            17
Unknown (Includes Not Applicable.. Unknown)      11
Black                                            10
MISSING                                          10
Other (Includes Not Applicable.. Unknown)         8
American Indian                                   3
Name: count, dtype: int64

In [18]:
cases['Race'].unique()

array(['Black(Non-Hispanic)', 'White Caucasian (Non-Hispanic)',
       'Hispanic', 'White Caucasian(Non-Hispanic)',
       'Black (Non-Hispanic)', 'Asian Or Pacific Islander',
       'Other(Includes Not Applicable.. Unknown)',
       'Other (Includes Not Applicable.. Unknown)', 'MISSING',
       'Unknown (Includes Not Applicable.. Unknown)', 'Black', 'White',
       'American Indian'], dtype=object)

In [21]:
replace_map = {
       'Black(Non-Hispanic)': 'Black',
       'White Caucasian (Non-Hispanic)': 'White',
       'Hispanic': 'Hispanic',
       'White Caucasian(Non-Hispanic)': 'White',
       'Black (Non-Hispanic)': 'Black',
       'Asian Or Pacific Islander': 'Asian',
       'Other(Includes Not Applicable.. Unknown)': 'Other',
       'Other (Includes Not Applicable.. Unknown)': 'Other',
       'MISSING': 'MISSING',
       'Unknown (Includes Not Applicable.. Unknown)': 'Unknown',
       'Black': 'Black',
       'White': 'White',
       'American Indian': 'American Indian'
       }
cases['Race'] = cases['Race'].replace(replace_map)
cases['Race'].value_counts()

Race
White              1452
Black              1185
Hispanic             88
Other                35
Asian                27
Unknown              11
MISSING              10
American Indian       3
Name: count, dtype: int64

In [22]:
cases.groupby('Race').agg({'Conviction': ['mean', 'count']}).reset_index().sort_values(('Conviction', 'mean'), ascending=False)

Race Conviction      
                         mean count
3         Hispanic   0.818182    88
4          MISSING   0.800000    10
1            Asian   0.740741    27
7            White   0.663223  1452
2            Black   0.610127  1185
5            Other   0.542857    35
0  American Indian   0.333333     3
6          Unknown   0.272727    11

In what localities (fips) are these disparities most severe?